# 02 — Assertions and Golden Outputs

## Why this notebook exists

In **`01_why_agents_are_hard_to_test.ipynb`** we saw how a naive `assert agent(x) == "expected"` unit test flakes the moment an agent varies its phrasing — and we named the three forces behind that fragility: non-determinism, no single ground truth, and multi-step compounding failures. The closing insight was: *"We can't `assert equals` our way out of this. We need graders that score outputs, not match them exactly."*

This notebook builds those graders — the cheapest, most reliable layer of the eval stack. Each grader is a plain callable that accepts an example and a candidate output and returns a `Score`. No LLM, no external service, no API key required.

## What you'll learn

- The `Score` dataclass — the shared return type every grader in this series produces.
- When **exact match** is appropriate (structured outputs, deterministic pipelines) and why it's brittle for free-text.
- How **contains** and **regex** graders give you partial-match power without LLM overhead.
- How to validate **structured outputs** with `pydantic` v2 — confirming an agent's JSON parses into the expected schema, and capturing the validation error when it doesn't.
- The discipline of **golden outputs**: storing a known-good string to a file and diffing against it — and why keeping goldens from silently going stale is as important as writing them.
- How all four grader families share a single `(example, output) -> Score` signature, so they compose transparently in notebook 03's harness.

## 1. Setup

All graders in this notebook share one return type: `Score`. Defining it here, once, as a `dataclass` gives us a consistent interface that notebook 03 will rely on when it builds the harness.

**Compatibility note:** In this notebook examples are plain dicts — `{"input": ..., "expected": ...}` — and graders read `example["expected"]`. Notebook 03 will formalize `Example` as a proper dataclass with `.input` and `.expected` attributes, at which point `example["expected"]` becomes `example.expected`. The grader signatures don't change — only how you access the field. This is called out again at each grader definition below.

If `pydantic` isn't installed yet:
```bash
pip install pydantic
```

In [ ]:
import re
import tempfile
from dataclasses import dataclass
from pathlib import Path

from pydantic import BaseModel, ValidationError
from typing import Literal

# Track every temp file written in this notebook for cleanup at the end.
_temp_files: list[Path] = []


@dataclass
class Score:
    key: str            # grader identifier, e.g. "exact_match"
    score: float        # normalized to [0.0, 1.0]
    passed: bool
    comment: str = ""


print("Setup OK")
print(f"Score fields: {[f.name for f in Score.__dataclass_fields__.values()]}")

## 2. Exact / Contains / Regex Matching

The three cheapest graders form a spectrum:

| Grader | Passes when | Best for |
|---|---|---|
| `exact_match` | `output == expected` exactly | Structured strings, enum labels, short deterministic answers |
| `make_contains(substring)` | `substring in output` | Checking that a required phrase or field name appears anywhere |
| `make_regex(pattern)` | `re.search(pattern, output)` matches | Format constraints — dates, codes, capitalization, numeric ranges |

**Exact match is brittle for free text** — a single trailing space or synonym flips it to fail. Its strength is precisely that brittleness: when an output *should* be deterministic (a classification label, a formatted code), exact match punishes any deviation.

Each grader accepts `(example, output)` where `example` is a dict `{"input": ..., "expected": ...}`. The `example["expected"]` field is used by `exact_match`; the contains/regex graders ignore `expected` and grade the raw output against the substring/pattern they were constructed with. In notebook 03, `example` becomes an `Example` dataclass and `example["expected"]` becomes `example.expected` — the grader body doesn't change.

In [ ]:
# ── Grader 1: exact_match ────────────────────────────────────────────────────

def exact_match(example: dict, output: str) -> Score:
    """Pass iff output equals example['expected'] exactly."""
    expected = example["expected"]
    passed = output == expected
    return Score(
        key="exact_match",
        score=1.0 if passed else 0.0,
        passed=passed,
        comment="" if passed else f"expected {expected!r}, got {output!r}",
    )


# ── Grader 2: make_contains ──────────────────────────────────────────────────

def make_contains(substring: str, key: str = "contains"):
    """Factory: return a grader that passes when `substring` appears in output."""
    def grader(example: dict, output: str) -> Score:
        passed = substring in output
        return Score(
            key=key,
            score=1.0 if passed else 0.0,
            passed=passed,
            comment="" if passed else f"{substring!r} not found in output",
        )
    grader.__name__ = key
    return grader


# ── Grader 3: make_regex ─────────────────────────────────────────────────────

def make_regex(pattern: str, key: str = "regex"):
    """Factory: return a grader that passes when `pattern` matches anywhere in output."""
    compiled = re.compile(pattern)
    def grader(example: dict, output: str) -> Score:
        match = compiled.search(output)
        passed = match is not None
        return Score(
            key=key,
            score=1.0 if passed else 0.0,
            passed=passed,
            comment="" if passed else f"pattern {pattern!r} not found in output",
        )
    grader.__name__ = key
    return grader


print("Graders defined: exact_match, make_contains, make_regex")

In [ ]:
### Try it

# Stubbed agent outputs — no API call needed.
output_correct   = "positive"
output_wrong     = "Positive"   # different capitalisation — exact_match will catch this
output_verbose   = "The sentiment is positive, with high confidence."

example = {"input": "The food was great!", "expected": "positive"}

# exact_match: pass and fail
s1 = exact_match(example, output_correct)
s2 = exact_match(example, output_wrong)
print(f"exact_match on {output_correct!r}: passed={s1.passed}, score={s1.score}")
print(f"exact_match on {output_wrong!r}:  passed={s2.passed}, comment={s2.comment!r}")
print()

# contains: checks that "positive" appears somewhere, even in a verbose answer
contains_positive = make_contains("positive", key="contains_positive")
s3 = contains_positive(example, output_verbose)
s4 = contains_positive(example, "The sentiment is negative.")
print(f"contains on verbose:   passed={s3.passed}")
print(f"contains on negative:  passed={s4.passed}, comment={s4.comment!r}")
print()

# regex: require the output to be exactly one of our label words (full-string match)
label_regex = make_regex(r"^(positive|negative|neutral)$", key="label_format")
s5 = label_regex(example, output_correct)
s6 = label_regex(example, output_verbose)
print(f"regex on {output_correct!r}:  passed={s5.passed}")
print(f"regex on verbose:     passed={s6.passed}, comment={s6.comment!r}")